# SN-01bis — Sirenisation Phase 1 bis (rejeu adresse historique)

**Proposition** : quand la Phase 1 n'aboutit pas à un SIREN
« VALIDE_FORT », avant de retenir le simple « VALIDE » ou de passer en Phase 2,
on retente la validation du SIREN déjà lié en comparant l'adresse PM FINESS
aux différentes adresses du siège disponibles dans SIRENE (adresses
historiques / alternatives).

**Pourquoi** : l'adresse de la PM dans FINESS n'a souvent pas été mise à jour,
alors que le siège du SIREN a déménagé dans SIRENE. L'ancienne adresse (celle de
FINESS) est en général encore présente dans SIRENE sous la forme d'un autre
établissement du même SIREN — souvent fermé (ancien siège).

**Méthode** : on reconstitue les adresses candidates à partir de tous les
établissements du SIREN (siège + secondaires + fermés), on rejoue le score
adresse, on garde le meilleur. Le score nom (indépendant de l'adresse) est
conservé de la Phase 1. On réutilise à l'identique `calc_score_global` et
`classifier_resultat`.

**Objectif** : vérifier que le volume partant en Phase 2 diminue.

Traitement indépendant du pipeline principal : rien n'est écrit dans les
fichiers que consomment les phases 2 et 3.

## 1. Imports

In [1]:
%pip install -r ../../requirements.txt --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from pathlib import Path

from src.connexion   import get_duckdb_connection
from src.phase1bis   import construire_adresses_alternatives, appliquer_phase1_bis
from src.display     import afficher_tableau
from config.settings import (
    FINESS_PM_CLEAN, SIRENE_UL_CLEAN, PARQUET_ETAB,
    SN_PHASE1BIS, RESULTS_SN_DIR, PROCESSED_DIR,
)

RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)

## 2. Rejouer la Phase 1

Relance de la Phase 1

In [3]:
from src.sirenisation import matching_direct_siren

df_pm = pd.read_parquet(FINESS_PM_CLEAN)
df_ul = pd.read_parquet(SIRENE_UL_CLEAN)
df_pm['siren_norm_pm'] = df_pm['siren_norm_pm'].fillna('').astype(str)
df_ul['siren']         = df_ul['siren'].astype(str)

df_p1 = matching_direct_siren(df_pm, df_ul)
print(f"Phase 1 : {len(df_p1):,} PM")
print(df_p1['statut'].value_counts().to_string())

PM traitées      : 56,193
   sans SIREN    : 2,090
   SIREN inconnu : 5,604
   à scorer      : 48,499


Phase 1 sirenisation:   0%|          | 0/48499 [00:00<?, ?it/s]

Phase 1 : 56,193 PM
statut
VALIDE_FORT      23085
VALIDE           15700
DOUTEUX           6870
SIREN_INCONNU     5604
REJETE            2844
SANS_SIREN        2090


## 3. Construire les adresses alternatives par SIREN

On charge tous les établissements SIRENE (actifs & fermés) via une vue
DuckDB dédiée, sans le filtre `etatAdministratifEtablissement = 'A'`

Pour limiter la mémoire, on ne garde que les SIREN présents dans la Phase 1.

In [4]:
# SIREN concernés (ceux liés en Phase 1)
sirens_utiles = set(df_p1['siren_ul'].dropna().astype(str).str.strip())
sirens_utiles.discard('')
sirens_utiles.discard('None')
print(f"SIREN à couvrir : {len(sirens_utiles):,}")

# Vue DuckDB : TOUS les établissements (actifs + fermés), colonnes d'adresse
con = get_duckdb_connection()
con.execute(f"""
    CREATE VIEW etab_all_adr AS
    SELECT siren,
           numeroVoieEtablissement, typeVoieEtablissement,
           libelleVoieEtablissement, codeCommuneEtablissement,
           etatAdministratifEtablissement, etablissementSiege
    FROM read_parquet('{PARQUET_ETAB}')
""")

df_sirens = pd.DataFrame({'siren': sorted(sirens_utiles)})
con.register('sirens_utiles', df_sirens)
df_etab_all = con.execute("""
    SELECT e.*
    FROM etab_all_adr e
    INNER JOIN sirens_utiles s ON e.siren = s.siren
""").df()
con.close()

print(f"Établissements récupérés : {len(df_etab_all):,}")
print("Répartition état :")
print(df_etab_all['etatAdministratifEtablissement'].value_counts().to_string())

SIREN à couvrir : 53,345


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Établissements récupérés : 294,389
Répartition état :
etatAdministratifEtablissement
A    171805
F    122584


In [5]:
df_adresses_alt = construire_adresses_alternatives(df_etab_all)
print(f"Adresses alternatives (lignes) : {len(df_adresses_alt):,}")
print(f"SIREN couverts                 : {df_adresses_alt['siren'].nunique():,}")

nb_par_siren = df_adresses_alt.groupby('siren').size()
print(f"Adresses par SIREN — médiane : {nb_par_siren.median():.0f}, max : {nb_par_siren.max()}")
afficher_tableau(df_adresses_alt.head(10), 'Aperçu adresses alternatives')

Adresses alternatives (lignes) : 264,983
SIREN couverts                 : 53,272
Adresses par SIREN — médiane : 2, max : 13064


siren,numero_voie_norm_ul,libelle_voie_complet_ul,code_commune_norm_ul,etat,est_siege
100148261,1,RUE VIVARAIS,54547,A,True
100148261,2,RUE LORD BYRON,06029,A,False
100148261,7,ALLEE BUGADIERES,06027,A,False
100148261,168,AVENUE SEMPER OLIVA,83090,A,False
100148261,47,RUE HOTEL VILLE,83113,A,False


## 4. Appliquer la Phase 1 bis

On rejoue le score adresse pour toutes les PM dont la Phase 1 n'a pas donné
VALIDE_FORT. Le rejeu ne peut qu'améliorer ou laisser inchangé (on garde le
meilleur des deux : score P1 ou meilleure adresse alternative).

In [6]:
df_bis = appliquer_phase1_bis(
    df_p1,
    df_adresses_alt,
    statuts_a_rejouer=('VALIDE', 'DOUTEUX', 'REJETE'),
)

n_rejoues   = int(df_bis['phase1bis_applique'].sum())
n_ameliores = int(df_bis['phase1bis_gain'].notna().sum())
print(f"PM rejouées            : {n_rejoues:,}")
print(f"PM dont statut change  : {n_ameliores:,}")

Phase 1 bis (rejeu adresse):   0%|          | 0/25414 [00:00<?, ?it/s]

PM rejouées            : 25,414
PM dont statut change  : 7,204


## 5. Détail des gains de statut

In [7]:
gains = df_bis[df_bis['phase1bis_gain'].notna()]
print("Transitions de statut (Phase 1 → Phase 1 bis) :")
print(gains['phase1bis_gain'].value_counts().to_string())

print("\nExemples de PM récupérées (adresse alternative trouvée) :")
cols_apercu = ['TX_Siren', 'raisonsociale_norm_pm', 'statut', 'statut_bis',
               'score_adresse', 'score_adresse_bis', 'adresse_alt_retenue',
               'etat_etab_retenu']
cols_apercu = [c for c in cols_apercu if c in gains.columns]
afficher_tableau(gains[cols_apercu].head(15), 'PM récupérées par Phase 1 bis')

Transitions de statut (Phase 1 → Phase 1 bis) :
phase1bis_gain
VALIDE → VALIDE_FORT     2627
DOUTEUX → VALIDE_FORT    1509
DOUTEUX → VALIDE         1210
REJETE → DOUTEUX          850
REJETE → VALIDE           846
REJETE → VALIDE_FORT      162

Exemples de PM récupérées (adresse alternative trouvée) :


TX_Siren,raisonsociale_norm_pm,statut,statut_bis,score_adresse,score_adresse_bis,adresse_alt_retenue,etat_etab_retenu
546120122,CLINIQUE SOUFFLE PONTET,DOUTEUX,VALIDE,7.110000,70.000000,RUE CHAPELLE (01185),F
260100193,MAISON RETRAITE PONT AIN,VALIDE,VALIDE_FORT,57.010000,81.440000,RUE CATHERINETTE (01304),A
339710733,CHATEAU VERNANGE,VALIDE,VALIDE_FORT,56.850000,81.220000,ROUTE MONTHIEUX (01333),A
260104518,MAPA CLAIRES FONTAINES SAINT VULBAS,VALIDE,VALIDE_FORT,70.000000,100.000000,RUE CLAIRES FONTAINES (01390),A
351473251,ASS ATELIER PROTEGE ST RAMBERT,REJETE,VALIDE,36.860000,100.000000,11 QUAI LUCIEN FRANC (01384),F


## 5 bis. Comparaison adresses (FINESS / siège Phase 1 / alternative retenue)

Pour rendre le gain lisible, on rapproche trois adresses :
- adresse_finess_pm : l'adresse de la PM dans FINESS
- adresse_siege_p1 : l'adresse du siège SIRENE utilisée en Phase 1
- adresse_alt_retenue : l'adresse alternative du SIREN (retenue parmi tous les etab)

In [8]:
if "adresse_complete_pm" in df_bis.columns:
    df_bis["adresse_finess_pm"] = (
        df_bis["adresse_complete_pm"].fillna("").astype(str).str.strip()
        + " " + df_bis.get("cdcommune_norm_pm", pd.Series("", index=df_bis.index))
                     .fillna("").astype(str).str.strip()
    ).str.strip()
else:
    df_bis["adresse_finess_pm"] = (
        df_bis.get("TX_NumeroVoie", "").fillna("").astype(str).str.strip() + " "
        + df_bis.get("TX_TypeVoie", "").fillna("").astype(str).str.strip() + " "
        + df_bis.get("TX_LibelleVoie", "").fillna("").astype(str).str.strip() + " "
        + df_bis.get("TX_CogCommune", "").fillna("").astype(str).str.strip()
    ).str.replace(r"\s+", " ", regex=True).str.strip()

cols_siege = [c for c in ["siren", "adresse_siege_complete_ul", "code_commune_norm_ul"]
              if c in df_ul.columns]
ul_siege = df_ul[cols_siege].drop_duplicates("siren").copy()
ul_siege["siren"] = ul_siege["siren"].astype(str)

df_bis["siren_ul"] = df_bis["siren_ul"].astype(str)
df_bis = df_bis.merge(ul_siege, left_on="siren_ul", right_on="siren",
                      how="left", suffixes=("", "_siege")).drop(columns=["siren"], errors="ignore")

df_bis["adresse_siege_p1"] = (
    df_bis.get("adresse_siege_complete_ul", pd.Series("", index=df_bis.index))
          .fillna("").astype(str).str.strip()
    + " " + df_bis.get("code_commune_norm_ul", pd.Series("", index=df_bis.index))
                 .fillna("").astype(str).str.strip()
).str.strip()

print("Adresses construites : adresse_finess_pm, adresse_siege_p1, adresse_alt_retenue")
apercu = df_bis[df_bis["phase1bis_gain"].notna()][
    ["TX_Siren", "adresse_finess_pm", "adresse_siege_p1", "adresse_alt_retenue"]
].head(10)
afficher_tableau(apercu, "Comparaison des adresses (PM récupérées)")

Adresses construites : adresse_finess_pm, adresse_siege_p1, adresse_alt_retenue


TX_Siren,adresse_finess_pm,adresse_siege_p1,adresse_alt_retenue
546120122,311 R DE LA CHAPELLE 01185,ALLEE DE RONCEVAUX 31561,RUE CHAPELLE (01185)
260100193,R CAEERINETTE 01304,RUE CATHERINETTE 01304,RUE CATHERINETTE (01304)
339710733,RTE DE MONEIEUX 01333,ROUTE DE MONTHIEUX 01333,ROUTE MONTHIEUX (01333)
260104518,R CLAIRES FONTAINES 01390,RUE CLAIRES FONTAINES 01390,RUE CLAIRES FONTAINES (01390)
351473251,11 QUAI LUCIEN FRANC 01384,26 RUE DE LA SCHAPPE 01384,11 QUAI LUCIEN FRANC (01384)


## 6. Mesure d'impact sur la Phase 2

Indicateur : le volume partant en Phase 2

In [9]:
def part_en_phase2(serie_statut):
    return ~serie_statut.isin(['VALIDE_FORT', 'VALIDE'])

n_p2_avant = int(part_en_phase2(df_bis['statut']).sum())
n_p2_apres = int(part_en_phase2(df_bis['statut_bis']).sum())
evites     = n_p2_avant - n_p2_apres

valides_avant = int(df_bis['statut'].isin(['VALIDE_FORT', 'VALIDE']).sum())
valides_apres = int(df_bis['statut_bis'].isin(['VALIDE_FORT', 'VALIDE']).sum())
fort_avant    = int((df_bis['statut'] == 'VALIDE_FORT').sum())
fort_apres    = int((df_bis['statut_bis'] == 'VALIDE_FORT').sum())

print("=" * 58)
print("IMPACT PHASE 1 BIS")
print("=" * 58)
print(f"  VALIDE_FORT     : {fort_avant:>7,}  →  {fort_apres:>7,}   (+{fort_apres-fort_avant:,})")
print(f"  Validés (total) : {valides_avant:>7,}  →  {valides_apres:>7,}   (+{valides_apres-valides_avant:,})")
print("-" * 58)
print(f"  Partent en P2   : {n_p2_avant:>7,}  →  {n_p2_apres:>7,}   (-{evites:,})")
print("=" * 58)
if n_p2_avant > 0:
    print(f"  Réduction du volume Phase 2 : {evites/n_p2_avant*100:.1f} %")

IMPACT PHASE 1 BIS
  VALIDE_FORT     :  23,085  →   27,383   (+4,298)
  Validés (total) :  38,785  →   42,512   (+3,727)
----------------------------------------------------------
  Partent en P2   :  17,408  →   13,681   (-3,727)
  Réduction du volume Phase 2 : 21.4 %


## 7. Export Excel

In [10]:
from src.excel_export import _style_header, _style_lignes, _auto_width

C_HEADER_SYNTH = "1F4E79"
C_HEADER_GAIN  = "1A7341"
C_LIGNES_GAIN  = "E8F5E9"
C_HEADER_FULL  = "1F4E79"
C_LIGNES_FULL  = "F2F6FC"

COLS_EXPORT = [
    "ID_PmSmsse", "NB_PmSmsseId", "TX_NumFinessPm", "dept_pm", "TX_DenominationPm",
    "nom_ul_retenu", "TX_Siren", "siren_ul",
    "score_nom", "score_adresse", "score_global", "statut",
    "score_adresse_bis", "score_global_bis", "statut_bis",
    "adresse_finess_pm", "adresse_siege_p1", "adresse_alt_retenue",
    "etat_etab_retenu", "phase1bis_applique", "phase1bis_gain",
]
cols = [c for c in COLS_EXPORT if c in df_bis.columns]

df_synth = pd.DataFrame([
    {"Indicateur": "VALIDE_FORT avant",         "Nombre": fort_avant},
    {"Indicateur": "VALIDE_FORT après",         "Nombre": fort_apres},
    {"Indicateur": "Validés total avant",       "Nombre": valides_avant},
    {"Indicateur": "Validés total après",       "Nombre": valides_apres},
    {"Indicateur": "—",                         "Nombre": ""},
    {"Indicateur": "Partent en Phase 2 avant",  "Nombre": n_p2_avant},
    {"Indicateur": "Partent en Phase 2 après",  "Nombre": n_p2_apres},
    {"Indicateur": "PM récupérées (P2 évités)", "Nombre": evites},
])

df_recuperes = df_bis[df_bis["phase1bis_gain"].notna()][cols]
df_complet   = df_bis[cols]

with pd.ExcelWriter(SN_PHASE1BIS, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthese_impact", index=False)
    ws = writer.sheets["Synthese_impact"]
    _style_header(ws, C_HEADER_SYNTH)
    from openpyxl.styles import Font, Alignment
    for row in ws.iter_rows(min_row=2):
        label = str(row[0].value)
        for cell in row:
            cell.alignment = Alignment(horizontal="left", vertical="center")
            if "évités" in label or "récupérées" in label:
                cell.font = Font(bold=True, name="Arial", size=10)
    _auto_width(ws, max_w=35)
    ws.freeze_panes = "A2"

    if len(df_recuperes) > 0:
        df_recuperes.to_excel(writer, sheet_name="PM_recuperees", index=False)
        ws = writer.sheets["PM_recuperees"]
        _style_header(ws, C_HEADER_GAIN)
        _style_lignes(ws, C_LIGNES_GAIN)
        _auto_width(ws)
        ws.freeze_panes = "A2"

    df_complet.to_excel(writer, sheet_name="Resultat_complet", index=False)
    ws = writer.sheets["Resultat_complet"]
    _style_header(ws, C_HEADER_FULL)
    _style_lignes(ws, C_LIGNES_FULL)
    _auto_width(ws)
    ws.freeze_panes = "A2"

print(f"Export OK → {SN_PHASE1BIS.resolve()}")
print(f"  Synthese_impact  : réduction Phase 2 = {evites:,}")
print(f"  PM_recuperees    : {len(df_recuperes):,} lignes")
print(f"  Resultat_complet : {len(df_complet):,} lignes")

Export OK → /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/sirenisation/sirenisation_phase1bis.xlsx
  Synthese_impact  : réduction Phase 2 = 3,727
  PM_recuperees    : 7,204 lignes
  Resultat_complet : 56,193 lignes
